In [39]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle
import shap
import matplotlib.pyplot as plt
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🎯 审稿人验证程序：SHAP分析（全部特征 - HTC）- 增强版")
print("="*80)
print("目的：计算所有特征的SHAP值，并查看RFE筛选的特征排名")
print("新增：Top 50蜂群图 (Beeswarm Plot) + 完整Excel数据导出")
print("="*80)

# ====================== 1. 定义模型结构（必须与训练时完全一致）======================
class ImprovedModel_Dynamic(nn.Module):
    """验证模型：动态特征数 - HTC"""
    def __init__(self, input_dim):
        super(ImprovedModel_Dynamic, self).__init__()
        
        # 第一隐藏层：128个神经元
        self.layer1 = nn.Linear(input_dim, 128)
        self.ln1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        
        # 第二隐藏层：64个神经元
        self.layer2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        
        # 第三隐藏层：32个神经元
        self.layer3 = nn.Linear(64, 32)
        self.ln3 = nn.LayerNorm(32)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.1)
        
        # 输出层
        self.layer4 = nn.Linear(32, 1)

    def forward(self, x):
        # 第一层
        x = self.layer1(x)
        x = self.ln1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        # 第二层
        x = self.layer2(x)
        x = self.ln2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        # 第三层
        x = self.layer3(x)
        x = self.ln3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        
        # 输出层
        x = self.layer4(x)
        return x

# ====================== 2. 定义路径 ======================
base_model_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\验证分析_AllFeatures_HTC"
base_shap_output_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\SHAP分析_AllFeatures_HTC1.17.2"

print(f"\n路径设置:")
print(f"  模型基础路径: {base_model_dir}")
print(f"  SHAP输出基础路径: {base_shap_output_dir}")

# ====================== 3. 🎯 RFE筛选出的特征（从原始HTC训练）======================
RFE_SELECTED_FEATURES_HTC = [
    "VEC",
    "am",
    "Mean_热膨胀(1/k)",
    "Mean_比热容J/g/k",
    "Mean_Pyykko(Triple Bond) (pm)",
    "Var_E13 Electron affinity"
]

print(f"\n🎯 RFE筛选的关键特征 (HTC):")
print(f"   当前列表中有 {len(RFE_SELECTED_FEATURES_HTC)} 个特征")
for i, feat in enumerate(RFE_SELECTED_FEATURES_HTC, 1):
    print(f"  {i}. {feat}")

print(f"\n" + "="*80)
print(f"⚠️ 请确认以上特征列表与实际HTC模型的RFE结果一致！")
print("="*80)

# ====================== 4. 加载数据 ======================
print("\n加载数据...")
df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx")

# 定义所有可能的特征
all_possible_features = [
    "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "Fe", "B", "V", 
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "Mean_A1 atomic number",
    "Mean_A6 atomic weight",
    "Mean_E2 electronegativity (Pauling)",
    "Mean_E5 energy ionization first",
    "Mean_E13 Electron affinity",
    "Mean_Electrophilicity Index",
    "Mean_Speed of Sound",
    "Mean_Neutron Cross Section",
    "Mean_Neutron Mass Absorption",
    "Mean_Space Group Number",
    "Mean_Glawe Number",
    "Mean_C1 temperature melting",
    "Mean_C2 temperature boiling",
    "Mean_density",
    "Mean_C3 enthalpy vaporization",
    "Mean_C4 enthalpy melting",
    "Mean_C5 enthalpy atomization",
    "Mean_热导率W/(mk)",
    "Mean_电导率(MS/m)",
    "Mean_电阻率(mΩ)",
    "Mean_热膨胀(1/k)",
    "Mean_比热容J/g/k",
    "Mean_摩尔热容(J/mol/k)",
    "Mean_摩尔体积(cm3/mol)",
    "Mean_莫氏硬度(MPa)",
    "Mean_covalent radius Cordero (pm)",
    "Mean_covalent radius Pyykko(Single Bond) (pm)",
    "Mean_covalent radius Pyykko(Double Bond) (pm)",
    "Mean_Pyykko(Triple Bond) (pm)",
    "Mean_S10 Lattice Constants a",
    "Mean_S11 Lattice Constants b",
    "Mean_S12 Lattice Constants c",
    "Mean_S13 radii atomic (coordination number 12) (pm)",
    "Mean_metallic radius(pm)",
    "Mean_metallic radius 12 Neighbors(pm)",
    
    "Var_A1 atomic number",
    "Var_A6 atomic weight",
    "Var_E2 electronegativity (Pauling)",
    "Var_E5 energy ionization first",
    "Var_E13 Electron affinity",
    "Var_Electrophilicity Index",
    "Var_Speed of Sound",
    "Var_Neutron Cross Section",
    "Var_Neutron Mass Absorption",
    "Var_Space Group Number",
    "Var_Glawe Number",
    "Var_C1 temperature melting",
    "Var_C2 temperature boiling",
    "Var_density",
    "Var_C3 enthalpy vaporization",
    "Var_C4 enthalpy melting",
    "Var_C5 enthalpy atomization",
    "Var_热导率W/(mk)",
    "Var_电导率(MS/m)",
    "Var_电阻率(mΩ)",
    "Var_热膨胀(1/k)",
    "Var_比热容J/g/k",
    "Var_摩尔热容(J/mol/k)",
    "Var_摩尔体积(cm3/mol)",
    "Var_莫氏硬度(MPa)",
    "Var_covalent radius Cordero (pm)",
    "Var_covalent radius Pyykko(Single Bond) (pm)",
    "Var_covalent radius Pyykko(Double Bond) (pm)",
    "Var_Pyykko(Triple Bond) (pm)",
    "Var_S10 Lattice Constants a",
    "Var_S11 Lattice Constants b",
    "Var_S12 Lattice Constants c",
    "Var_S13 radii atomic (coordination number 12) (pm)",
    "Var_metallic radius(pm)",
    "Var_metallic radius 12 Neighbors(pm)",
]

# 只使用数据中实际存在的特征
features_all = [col for col in all_possible_features if col in df.columns]
actual_feature_count = len(features_all)

print(f"  数据中实际存在的特征数: {actual_feature_count}")

# 更新路径
model_dir = base_model_dir
shap_output_dir = base_shap_output_dir
if not os.path.exists(shap_output_dir):
    os.makedirs(shap_output_dir)

# 尝试加载特征列表文件
features_file = os.path.join(model_dir, f'features_{actual_feature_count}_HTC.txt')
if os.path.exists(features_file):
    print(f"  ✅ 从文件加载特征列表: {features_file}")
    with open(features_file, 'r', encoding='utf-8') as f:
        features_all = [line.strip() for line in f.readlines()]
    actual_feature_count = len(features_all)
else:
    print(f"  ⚠️ 未找到特征列表文件，使用数据中的特征")

print(f"  最终使用特征数: {actual_feature_count}")

# 加载标准化器
scaler_path = os.path.join(model_dir, f'scaler_{actual_feature_count}features_HTC.pkl')
if not os.path.exists(scaler_path):
    print(f"  ❌ 错误：找不到标准化器文件: {scaler_path}")
    print(f"  请先运行训练程序！")
    exit(1)

with open(scaler_path, 'rb') as f:
    scaler_all = pickle.load(f)
print(f"  ✅ 标准化器已加载")

# ====================== 5. 加载模型 ======================
print("\n加载模型...")
device = torch.device("cpu")

model = ImprovedModel_Dynamic(input_dim=actual_feature_count)

model_weights_path = os.path.join(model_dir, f'model_{actual_feature_count}features_weights_HTC.pth')
if not os.path.exists(model_weights_path):
    print(f"  ❌ 错误：找不到模型权重文件: {model_weights_path}")
    print(f"  请先运行训练程序！")
    exit(1)

model.load_state_dict(torch.load(model_weights_path, map_location='cpu'))
print(f"  ✅ 模型权重已加载")
print(f"  模型使用 {actual_feature_count} 个特征")

model.to(device)
model.eval()

# ====================== 6. 准备数据 ======================
print("\n准备SHAP分析数据...")
X_original = df[features_all].values
X_scaled = scaler_all.transform(X_original)
y = df['high temperature compression'].values

print(f"  样本数: {X_scaled.shape[0]}")
print(f"  特征数: {X_scaled.shape[1]}")

# ====================== 7. 创建SHAP模型包装器 ======================
class ShapModelWrapper:
    """SHAP模型包装器"""
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
    def __call__(self, X):
        if isinstance(X, np.ndarray):
            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        else:
            X_tensor = X
            
        with torch.no_grad():
            output = self.model(X_tensor)
            
        return output.cpu().numpy()

# ====================== 8. SHAP分析 - 使用Kernel解释器 ======================
print("\n开始SHAP分析...")
print("  这可能需要几分钟时间...")

max_bg_samples = min(20, X_scaled.shape[0])
bg_indices = np.random.choice(X_scaled.shape[0], max_bg_samples, replace=False)
bg_data = X_scaled[bg_indices]
print(f"  背景样本数: {max_bg_samples}")

model_wrapper = ShapModelWrapper(model)
explainer = shap.KernelExplainer(model_wrapper, bg_data)
print(f"  ✅ SHAP解释器已创建")

print(f"  正在计算所有 {X_scaled.shape[0]} 个样本的SHAP值...")
shap_values = explainer.shap_values(X_scaled)

# 处理SHAP值格式
if isinstance(shap_values, list) and len(shap_values) == 1:
    shap_values = shap_values[0]

if len(np.array(shap_values).shape) > 2:
    shap_values = np.squeeze(shap_values)

print(f"  ✅ SHAP值计算完成")
print(f"  SHAP值形状: {np.array(shap_values).shape}")

# ====================== 9. 计算特征重要性 ======================
print("\n计算特征重要性...")
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

shap_importance_df = pd.DataFrame({
    'Feature': features_all,
    'Mean_Absolute_SHAP_Value': mean_abs_shap,
    'Relative_Importance_%': mean_abs_shap / np.sum(mean_abs_shap) * 100
})

shap_importance_df = shap_importance_df.sort_values('Mean_Absolute_SHAP_Value', ascending=False)
shap_importance_df['Rank'] = range(1, len(shap_importance_df) + 1)

importance_path = os.path.join(shap_output_dir, f'feature_importance_all_{actual_feature_count}_HTC.xlsx')
shap_importance_df.to_excel(importance_path, index=False)
print(f"  ✅ 特征重要性已保存: {importance_path}")

# ====================== 10. 🎯 关键分析：RFE的特征排在第几？======================
print("\n" + "="*80)
print(f"🎯 关键结果：RFE筛选的特征在{actual_feature_count}个特征中的SHAP排名 (HTC)")
print("="*80)

rfe_ranking_results = []
features_not_found = []

for rfe_feature in RFE_SELECTED_FEATURES_HTC:
    feature_row = shap_importance_df[shap_importance_df['Feature'] == rfe_feature]
    
    if not feature_row.empty:
        rank = int(feature_row['Rank'].values[0])
        shap_value = float(feature_row['Mean_Absolute_SHAP_Value'].values[0])
        relative_imp = float(feature_row['Relative_Importance_%'].values[0])
        
        rfe_ranking_results.append({
            'RFE_Feature': rfe_feature,
            'SHAP_Rank': rank,
            'Mean_Abs_SHAP': shap_value,
            'Relative_Importance_%': relative_imp
        })
        
        print(f"✓ {rfe_feature}")
        print(f"  排名: {rank} / {actual_feature_count}")
        print(f"  平均绝对SHAP值: {shap_value:.6f}")
        print(f"  相对重要性: {relative_imp:.2f}%")
        print()
    else:
        features_not_found.append(rfe_feature)
        print(f"⚠️ 未找到特征: {rfe_feature}")

if rfe_ranking_results:
    rfe_ranking_df = pd.DataFrame(rfe_ranking_results)
    rfe_ranking_df = rfe_ranking_df.sort_values('SHAP_Rank')
    rfe_ranking_path = os.path.join(shap_output_dir, f'RFE_features_SHAP_ranking_HTC.xlsx')
    rfe_ranking_df.to_excel(rfe_ranking_path, index=False)
    print(f"\n✅ RFE特征排名已保存: {rfe_ranking_path}")

    # 统计分析
    print("\n" + "="*80)
    print("📊 统计总结 (HTC)")
    print("="*80)
    ranks = [r['SHAP_Rank'] for r in rfe_ranking_results]
    print(f"找到的RFE特征数: {len(ranks)} / {len(RFE_SELECTED_FEATURES_HTC)}")
    print(f"\nRFE特征排名统计:")
    print(f"  最高排名: {min(ranks)}")
    print(f"  最低排名: {max(ranks)}")
    print(f"  平均排名: {np.mean(ranks):.1f}")
    print(f"  中位排名: {np.median(ranks):.1f}")

    top_10_count = sum(1 for r in ranks if r <= 10)
    top_20_count = sum(1 for r in ranks if r <= 20)
    top_30_count = sum(1 for r in ranks if r <= 30)

    print(f"\n排名分布:")
    print(f"  Top 10: {top_10_count} / {len(ranks)} 特征 ({top_10_count/len(ranks)*100:.1f}%)")
    print(f"  Top 20: {top_20_count} / {len(ranks)} 特征 ({top_20_count/len(ranks)*100:.1f}%)")
    print(f"  Top 30: {top_30_count} / {len(ranks)} 特征 ({top_30_count/len(ranks)*100:.1f}%)")

# ====================== 11. 🔥🔥🔥 新增：创建Top 50蜂群图数据 ======================
print("\n" + "="*80)
print("🔥 准备 Top 50 蜂群图 (Beeswarm Plot) 数据...")
print("="*80)

# 创建用于蜂群图的完整数据Excel（供Origin等软件使用）
print("  创建Top 50蜂群图数据表...")

# Sheet 1: 完整SHAP值矩阵（样本 × 特征）
shap_values_df_full = pd.DataFrame(
    shap_values,
    columns=features_all
)
shap_values_df_full.insert(0, 'Sample_Index', range(len(shap_values_df_full)))

# Sheet 2: 完整特征值矩阵（样本 × 特征）
feature_values_df_full = pd.DataFrame(
    X_original,
    columns=features_all
)
feature_values_df_full.insert(0, 'Sample_Index', range(len(feature_values_df_full)))

# Sheet 3: 🔥🔥🔥 Top 50特征的蜂群图数据（长格式）
top_50_features = shap_importance_df.head(50)['Feature'].tolist()
top_50_indices = [features_all.index(f) for f in top_50_features]

beeswarm_data_list = []
for sample_idx in range(len(X_scaled)):
    for feat_idx, feat_name in zip(top_50_indices, top_50_features):
        beeswarm_data_list.append({
            'Sample_Index': sample_idx,
            'Feature': feat_name,
            'Feature_Rank': top_50_features.index(feat_name) + 1,  # 添加排名信息
            'Feature_Value': X_original[sample_idx, feat_idx],
            'Feature_Value_Normalized': X_scaled[sample_idx, feat_idx],
            'SHAP_Value': shap_values[sample_idx, feat_idx],
            'Absolute_SHAP_Value': abs(shap_values[sample_idx, feat_idx]),
            'Is_RFE_Selected': feat_name in RFE_SELECTED_FEATURES_HTC  # 标记RFE特征
        })

beeswarm_df = pd.DataFrame(beeswarm_data_list)

# Sheet 4: 统计汇总（Top 50每个特征的统计信息）
stats_list = []
for feat in top_50_features:
    feat_idx = features_all.index(feat)
    feat_shap = shap_values[:, feat_idx]
    feat_vals = X_original[:, feat_idx]
    
    stats_list.append({
        'Feature': feat,
        'Rank': int(shap_importance_df[shap_importance_df['Feature']==feat]['Rank'].values[0]),
        'Is_RFE_Selected': feat in RFE_SELECTED_FEATURES_HTC,
        'Mean_Abs_SHAP': np.mean(np.abs(feat_shap)),
        'Std_SHAP': np.std(feat_shap),
        'Min_SHAP': np.min(feat_shap),
        'Max_SHAP': np.max(feat_shap),
        'Median_SHAP': np.median(feat_shap),
        'Q25_SHAP': np.percentile(feat_shap, 25),
        'Q75_SHAP': np.percentile(feat_shap, 75),
        'Mean_Feature_Value': np.mean(feat_vals),
        'Std_Feature_Value': np.std(feat_vals),
        'Min_Feature_Value': np.min(feat_vals),
        'Max_Feature_Value': np.max(feat_vals),
        'Relative_Importance_%': float(shap_importance_df[shap_importance_df['Feature']==feat]['Relative_Importance_%'].values[0])
    })

stats_df = pd.DataFrame(stats_list)

# 保存到Excel（多个Sheet）
beeswarm_excel_path = os.path.join(shap_output_dir, 'beeswarm_plot_data_TOP50_complete_HTC.xlsx')
with pd.ExcelWriter(beeswarm_excel_path) as writer:
    # Sheet 0: 说明
    readme_data = pd.DataFrame({
        '说明': [
            '📊 Top 50蜂群图完整数据集',
            '',
            '=' * 60,
            'Sheet结构说明：',
            '=' * 60,
            '',
            'Sheet 1: SHAP_Values_Full',
            '  - 所有样本 × 所有特征的SHAP值矩阵',
            '  - 用途：完整SHAP分析',
            '',
            'Sheet 2: Feature_Values_Full',
            '  - 所有样本 × 所有特征的原始值矩阵',
            '  - 用途：特征值分析',
            '',
            'Sheet 3: Beeswarm_Data_Top50 ⭐⭐⭐',
            '  - Top 50特征的蜂群图数据（长格式）',
            '  - 用途：直接用于Origin/Python绘制蜂群图',
            '  - 包含：Sample_Index, Feature, Feature_Rank, Feature_Value,',
            '         Feature_Value_Normalized, SHAP_Value, Absolute_SHAP_Value,',
            '         Is_RFE_Selected',
            '',
            'Sheet 4: Statistics_Top50',
            '  - Top 50特征的统计信息汇总',
            '  - 包含：排名、SHAP统计量、特征值统计量',
            '',
            '=' * 60,
            'Origin绘图步骤：',
            '=' * 60,
            '',
            '1. 导入 Sheet 3 (Beeswarm_Data_Top50)',
            '2. 创建散点图：',
            '   - X轴：SHAP_Value',
            '   - Y轴：Feature',
            '   - 颜色：Feature_Value_Normalized',
            '   - 大小：Absolute_SHAP_Value（可选）',
            '3. 设置颜色映射：Coolwarm 或 RdYlBu',
            '4. 添加垂直线：x=0（红色虚线）',
            '5. 标记RFE特征：Is_RFE_Selected=True',
            '',
            '=' * 60,
            '数据信息：',
            '=' * 60,
            f'生成时间: {pd.Timestamp.now()}',
            f'样本数: {len(X_scaled)}',
            f'总特征数: {actual_feature_count}',
            f'Top特征数: 50',
            f'RFE特征数: {len(RFE_SELECTED_FEATURES_HTC)}',
            '',
            'RFE筛选的特征：',
        ] + [f'  {i+1}. {feat}' for i, feat in enumerate(RFE_SELECTED_FEATURES_HTC)]
    })
    readme_data.to_excel(writer, sheet_name='0_README', index=False)
    
    # 其他数据Sheet
    shap_values_df_full.to_excel(writer, sheet_name='1_SHAP_Values_Full', index=False)
    feature_values_df_full.to_excel(writer, sheet_name='2_Feature_Values_Full', index=False)
    beeswarm_df.to_excel(writer, sheet_name='3_Beeswarm_Data_Top50', index=False)
    stats_df.to_excel(writer, sheet_name='4_Statistics_Top50', index=False)

print(f"  ✅ Top 50蜂群图完整数据已保存: {beeswarm_excel_path}")
print(f"     包含 {len(X_scaled)} 个样本 × Top 50 个特征")
print(f"     数据点总数: {len(beeswarm_df)} 个")

# ====================== 12. 🔥🔥🔥 新增：绘制Top 50蜂群图 ======================
print("\n创建Top 50蜂群图可视化...")

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 🔥 图1：经典SHAP蜂群图（Top 50）
print("  绘制经典SHAP蜂群图（Top 50）...")

try:
    # 创建shap.Explanation对象
    shap_values_top50 = shap_values[:, top_50_indices]
    X_display_top50 = X_scaled[:, top_50_indices]
    
    plt.figure(figsize=(12, 20))
    
    explanation = shap.Explanation(
        values=shap_values_top50,
        base_values=np.full(len(shap_values_top50), explainer.expected_value),
        data=X_display_top50,
        feature_names=top_50_features
    )
    
    # 绘制蜂群图
    shap.plots.beeswarm(explanation, max_display=50, show=False)
    
    plt.title('Top 50 Features - SHAP Beeswarm Plot (HTC)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(shap_output_dir, 'beeswarm_top50_classic_HTC.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✅ 经典蜂群图已保存 (Top 50)")
    
except Exception as e:
    print(f"  ⚠️ 经典蜂群图绘制失败: {e}")
    print(f"     将继续使用自定义绘图方法")

# 🔥 图2：自定义蜂群图（Top 50，★标记RFE特征）
print("  绘制自定义蜂群图（Top 50，带RFE标记）...")

fig, ax = plt.subplots(figsize=(12, 20))

# 准备绘图数据
positions = []
shap_vals = []
feature_vals_normalized = []

for i, (feat_idx, feat_name) in enumerate(zip(top_50_indices, top_50_features)):
    feat_shap = shap_values[:, feat_idx]
    feat_val = X_scaled[:, feat_idx]  # 使用标准化值用于颜色映射
    
    positions.extend([i] * len(feat_shap))
    shap_vals.extend(feat_shap)
    feature_vals_normalized.extend(feat_val)

# 创建散点图（模拟蜂群效果）
scatter = ax.scatter(
    shap_vals, 
    positions,
    c=feature_vals_normalized,
    cmap='coolwarm',
    alpha=0.6,
    s=15,
    edgecolors='none'
)

# 标记RFE特征
for i, feat_name in enumerate(top_50_features):
    if feat_name in RFE_SELECTED_FEATURES_HTC:
        # 添加金色星标
        ax.text(
            ax.get_xlim()[1] * 0.95, 
            i, 
            '★', 
            fontsize=12, 
            color='gold',
            ha='right',
            va='center',
            fontweight='bold',
            bbox=dict(boxstyle='circle', facecolor='red', alpha=0.3, edgecolor='gold', linewidth=2)
        )

# 设置y轴
ax.set_yticks(range(len(top_50_features)))
ax.set_yticklabels(top_50_features, fontsize=9)
ax.set_ylim(-0.5, len(top_50_features) - 0.5)

# 设置x轴
ax.set_xlabel('SHAP value (impact on model output)', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.4)

# 添加颜色条
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Feature Value\n(Standardized)', fontsize=11, rotation=270, labelpad=25)

# 设置标题
ax.set_title('Top 50 Features - SHAP Beeswarm Plot (HTC)\n★ = RFE Selected Features', 
             fontsize=16, fontweight='bold', pad=20)

# 添加网格
ax.grid(axis='x', alpha=0.3, linestyle='--')

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='gold', alpha=0.6, edgecolor='red', linewidth=2, 
          label=f'★ RFE Selected ({len(RFE_SELECTED_FEATURES_HTC)} features)'),
    Patch(facecolor='blue', label='High Feature Value'),
    Patch(facecolor='red', label='Low Feature Value')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(shap_output_dir, 'beeswarm_top50_custom_HTC.png'), 
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✅ 自定义蜂群图已保存 (Top 50, 带RFE★标记)")

# 🔥 图3：简化版蜂群图（仅RFE特征）
if rfe_ranking_results:
    print("  绘制RFE特征蜂群图...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 按SHAP排名排序RFE特征
    rfe_features_sorted = [r['RFE_Feature'] for r in sorted(rfe_ranking_results, 
                                                              key=lambda x: x['SHAP_Rank'])]
    rfe_indices_sorted = [features_all.index(f) for f in rfe_features_sorted]
    
    positions = []
    shap_vals = []
    feature_vals_normalized = []
    
    for i, feat_idx in enumerate(rfe_indices_sorted):
        feat_shap = shap_values[:, feat_idx]
        feat_val = X_scaled[:, feat_idx]
        
        positions.extend([i] * len(feat_shap))
        shap_vals.extend(feat_shap)
        feature_vals_normalized.extend(feat_val)
    
    scatter = ax.scatter(
        shap_vals,
        positions,
        c=feature_vals_normalized,
        cmap='RdYlBu_r',
        alpha=0.7,
        s=60,
        edgecolors='black',
        linewidths=0.8
    )
    
    ax.set_yticks(range(len(rfe_features_sorted)))
    ax.set_yticklabels(rfe_features_sorted, fontsize=12, fontweight='bold')
    ax.set_ylim(-0.5, len(rfe_features_sorted) - 0.5)
    
    ax.set_xlabel('SHAP value (impact on model output)', fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.6)
    
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
    cbar.set_label('Feature Value\n(Standardized)', fontsize=11, rotation=270, labelpad=20)
    
    ax.set_title('RFE Selected Features - SHAP Beeswarm Plot (HTC)', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(os.path.join(shap_output_dir, 'beeswarm_RFE_only_HTC.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✅ RFE特征蜂群图已保存")

# ====================== 13. 原有可视化（Top 50柱状图等）======================
print("\n创建传统可视化图表...")

# 图1：特征重要性排名（突出显示RFE特征）
fig, ax = plt.subplots(figsize=(14, 20))

num_to_show = min(50, len(shap_importance_df))
colors = ['red' if feat in RFE_SELECTED_FEATURES_HTC else 'gray' 
          for feat in shap_importance_df['Feature'].head(num_to_show)]

ax.barh(range(num_to_show), 
        shap_importance_df['Mean_Absolute_SHAP_Value'].head(num_to_show), 
        color=colors, alpha=0.6)

ax.set_yticks(range(num_to_show))
ax.set_yticklabels(shap_importance_df['Feature'].head(num_to_show), fontsize=9)
ax.set_xlabel('Mean Absolute SHAP Value', fontsize=13)
ax.set_ylabel('Features', fontsize=13)
ax.set_title(f'Top {num_to_show} Feature Importance - HTC (Red = RFE Selected)', 
             fontsize=15, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.6, label=f'RFE Selected ({len(RFE_SELECTED_FEATURES_HTC)} features)'),
    Patch(facecolor='gray', alpha=0.6, label='Other Features')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(shap_output_dir, 'feature_importance_ranking_top50_HTC.png'), 
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✅ Top 50特征重要性排名图已保存")

# 图2：RFE特征的排名条形图
if rfe_ranking_results:
    fig, ax = plt.subplots(figsize=(10, 6))

    rfe_sorted = rfe_ranking_df.sort_values('SHAP_Rank')
    bars = ax.barh(rfe_sorted['RFE_Feature'], rfe_sorted['SHAP_Rank'], 
                   color='steelblue', alpha=0.7, edgecolor='black')

    for i, (idx, row) in enumerate(rfe_sorted.iterrows()):
        ax.text(row['SHAP_Rank'] + 1, i, f"#{int(row['SHAP_Rank'])}", 
                va='center', fontsize=11, fontweight='bold')

    ax.set_xlabel(f'Rank (out of {actual_feature_count} features)', fontsize=12)
    ax.set_ylabel('RFE Selected Features', fontsize=12)
    ax.set_title('SHAP Ranking of RFE Selected Features - HTC', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(0, max(rfe_sorted['SHAP_Rank']) + 10)

    plt.tight_layout()
    plt.savefig(os.path.join(shap_output_dir, 'RFE_features_ranking_HTC.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✅ RFE特征排名图已保存")

# ====================== 14. 保存详细的SHAP值数据 ======================
print("\n保存详细SHAP数据...")

base_value = explainer.expected_value
if hasattr(base_value, "__len__") and len(base_value) == 1:
    base_value = base_value[0]

with torch.no_grad():
    all_predictions = model(torch.tensor(X_scaled, dtype=torch.float32).to(device)).cpu().numpy().flatten()

feature_values_df = pd.DataFrame(X_original, columns=features_all)
shap_values_df = pd.DataFrame(shap_values, columns=[f'SHAP_{col}' for col in features_all])
predictions_df = pd.DataFrame({
    'sample_index': range(len(X_scaled)),
    'prediction': all_predictions,
    'actual_value': y,
    'base_value': base_value
})

combined_df = pd.concat([predictions_df, feature_values_df, shap_values_df], axis=1)
combined_path = os.path.join(shap_output_dir, f'all_features_shap_predictions_HTC.xlsx')
combined_df.to_excel(combined_path, index=False)
print(f"  ✅ 完整SHAP数据已保存: {combined_path}")

# ====================== 15. 生成汇总报告 ======================
print("\n生成汇总报告...")

report_path = os.path.join(shap_output_dir, 'SHAP_Analysis_Report_HTC.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write(f"审稿人验证分析：{actual_feature_count}特征SHAP分析报告 (HTC) - Top 50增强版\n")
    f.write("="*80 + "\n\n")
    
    f.write("分析目的:\n")
    f.write("  验证RFE筛选的特征在全部特征中的重要性排名\n\n")
    
    f.write("数据信息:\n")
    f.write(f"  总特征数: {actual_feature_count}\n")
    f.write(f"  样本数: {X_scaled.shape[0]}\n")
    f.write(f"  RFE筛选特征数: {len(RFE_SELECTED_FEATURES_HTC)}\n")
    f.write(f"  找到的RFE特征数: {len(rfe_ranking_results)}\n")
    f.write(f"  Top特征展示数: 50\n\n")
    
    if rfe_ranking_results:
        f.write("="*80 + "\n")
        f.write("关键结果：RFE筛选的特征在SHAP分析中的排名\n")
        f.write("="*80 + "\n\n")
        
        for result in rfe_ranking_results:
            f.write(f"特征: {result['RFE_Feature']}\n")
            f.write(f"  SHAP排名: {result['SHAP_Rank']} / {actual_feature_count}\n")
            f.write(f"  平均绝对SHAP值: {result['Mean_Abs_SHAP']:.6f}\n")
            f.write(f"  相对重要性: {result['Relative_Importance_%']:.2f}%\n")
            if result['SHAP_Rank'] <= 50:
                f.write(f"  ✓ 在Top 50内\n")
            f.write("\n")
        
        f.write("="*80 + "\n")
        f.write("统计总结\n")
        f.write("="*80 + "\n\n")
        ranks = [r['SHAP_Rank'] for r in rfe_ranking_results]
        f.write(f"找到的RFE特征数: {len(ranks)} / {len(RFE_SELECTED_FEATURES_HTC)}\n\n")
        f.write(f"RFE特征排名统计:\n")
        f.write(f"  最高排名: {min(ranks)}\n")
        f.write(f"  最低排名: {max(ranks)}\n")
        f.write(f"  平均排名: {np.mean(ranks):.1f}\n")
        f.write(f"  中位排名: {np.median(ranks):.1f}\n\n")
        
        top_50_count = sum(1 for r in ranks if r <= 50)
        f.write(f"排名分布:\n")
        f.write(f"  Top 10: {top_10_count} / {len(ranks)} 特征 ({top_10_count/len(ranks)*100:.1f}%)\n")
        f.write(f"  Top 20: {top_20_count} / {len(ranks)} 特征 ({top_20_count/len(ranks)*100:.1f}%)\n")
        f.write(f"  Top 30: {top_30_count} / {len(ranks)} 特征 ({top_30_count/len(ranks)*100:.1f}%)\n")
        f.write(f"  Top 50: {top_50_count} / {len(ranks)} 特征 ({top_50_count/len(ranks)*100:.1f}%)\n\n")
        
        f.write("="*80 + "\n")
        f.write("结论\n")
        f.write("="*80 + "\n\n")
        
        if np.mean(ranks) <= 20:
            f.write("✅ RFE筛选的特征表现优异！\n")
            f.write("   平均排名在Top 20以内，证明RFE筛选方法有效。\n")
        elif np.mean(ranks) <= 30:
            f.write("✓ RFE筛选的特征表现良好。\n")
            f.write("   平均排名在Top 30以内，RFE筛选方法合理。\n")
        else:
            f.write("⚠️ RFE筛选的特征排名较为分散。\n")
            f.write("   建议进一步分析特征筛选策略。\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("生成的文件:\n")
    f.write("="*80 + "\n")
    f.write(f"1. feature_importance_all_{actual_feature_count}_HTC.xlsx - 全部特征SHAP重要性\n")
    f.write("2. RFE_features_SHAP_ranking_HTC.xlsx - RFE特征排名\n")
    f.write("3. beeswarm_plot_data_TOP50_complete_HTC.xlsx - 🔥 Top 50蜂群图完整数据（4个Sheet）\n")
    f.write("4. beeswarm_top50_classic_HTC.png - 🔥 经典蜂群图（Top 50）\n")
    f.write("5. beeswarm_top50_custom_HTC.png - 🔥 自定义蜂群图（Top 50，带RFE★标记）\n")
    f.write("6. beeswarm_RFE_only_HTC.png - 🔥 RFE特征蜂群图\n")
    f.write("7. feature_importance_ranking_top50_HTC.png - 特征重要性柱状图（Top 50）\n")
    f.write("8. RFE_features_ranking_HTC.png - RFE特征排名条形图\n")
    f.write("9. all_features_shap_predictions_HTC.xlsx - 完整SHAP预测数据\n")
    f.write("10. SHAP_Analysis_Report_HTC.txt - 本报告\n")
    f.write("="*80 + "\n")

print(f"  ✅ 汇总报告已保存: {report_path}")

# ====================== 16. 最终输出 ======================
print("\n" + "="*80)
print("=== SHAP分析完成 (HTC) - Top 50增强版 ===")
print("="*80)
print(f"所有结果已保存到：{shap_output_dir}")
print(f"\n模型使用了 {actual_feature_count} 个特征")
print("\n🔥 Top 50 新增内容:")
print("  ✓ Top 50蜂群图可视化（3种风格）")
print("  ✓ Top 50完整蜂群图数据Excel（4个Sheet，可用于Origin）")
print("\n主要结果文件:")
print(f"  1. beeswarm_plot_data_TOP50_complete_HTC.xlsx - 🔥 Top 50蜂群图数据（⭐ 可直接导入Origin）")
print(f"  2. beeswarm_top50_custom_HTC.png - 🔥 Top 50自定义蜂群图（带★标记）")
print(f"  3. beeswarm_top50_classic_HTC.png - 🔥 Top 50经典蜂群图")
print(f"  4. RFE_features_SHAP_ranking_HTC.xlsx - RFE特征排名")
print(f"  5. feature_importance_all_{actual_feature_count}_HTC.xlsx - 全部特征重要性")
print(f"  6. SHAP_Analysis_Report_HTC.txt - 汇总报告")
print("\n" + "="*80)
print("🎯 审稿人验证完成！Top 50特征分析已就绪！")
print("="*80)

🎯 审稿人验证程序：SHAP分析（全部特征 - HTC）- 增强版
目的：计算所有特征的SHAP值，并查看RFE筛选的特征排名
新增：Top 50蜂群图 (Beeswarm Plot) + 完整Excel数据导出

路径设置:
  模型基础路径: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\验证分析_AllFeatures_HTC
  SHAP输出基础路径: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\SHAP分析_AllFeatures_HTC1.17.2

🎯 RFE筛选的关键特征 (HTC):
   当前列表中有 6 个特征
  1. VEC
  2. am
  3. Mean_热膨胀(1/k)
  4. Mean_比热容J/g/k
  5. Mean_Pyykko(Triple Bond) (pm)
  6. Var_E13 Electron affinity

⚠️ 请确认以上特征列表与实际HTC模型的RFE结果一致！

加载数据...
  数据中实际存在的特征数: 96
  ✅ 从文件加载特征列表: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\验证分析_AllFeatures_HTC\features_96_HTC.txt
  最终使用特征数: 96
  ✅ 标准化器已加载

加载模型...
  ✅ 模型权重已加载
  模型使用 96 个特征

准备SHAP分析数据...
  样本数: 34
  特征数: 96

开始SHAP分析...
  这可能需要几分钟时间...
  背景样本数: 20
  ✅ SHAP解释器已创建
  正在计算所有 34 个样本的SHAP值...


  0%|          | 0/34 [00:00<?, ?it/s]

  ✅ SHAP值计算完成
  SHAP值形状: (34, 96)

计算特征重要性...
  ✅ 特征重要性已保存: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\SHAP分析_AllFeatures_HTC1.17.2\feature_importance_all_96_HTC.xlsx

🎯 关键结果：RFE筛选的特征在96个特征中的SHAP排名 (HTC)
✓ VEC
  排名: 15 / 96
  平均绝对SHAP值: 1.568535
  相对重要性: 2.24%

✓ am
  排名: 33 / 96
  平均绝对SHAP值: 0.490705
  相对重要性: 0.70%

✓ Mean_热膨胀(1/k)
  排名: 1 / 96
  平均绝对SHAP值: 6.501742
  相对重要性: 9.27%

✓ Mean_比热容J/g/k
  排名: 38 / 96
  平均绝对SHAP值: 0.361359
  相对重要性: 0.51%

✓ Mean_Pyykko(Triple Bond) (pm)
  排名: 56 / 96
  平均绝对SHAP值: 0.119649
  相对重要性: 0.17%

✓ Var_E13 Electron affinity
  排名: 21 / 96
  平均绝对SHAP值: 1.333073
  相对重要性: 1.90%


✅ RFE特征排名已保存: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充_improved\SHAP分析_AllFeatures_HTC1.17.2\RFE_features_SHAP_ranking_HTC.xlsx

📊 统计总结 (HTC)
找到的RFE特征数: 6 / 6

RFE特征排名统计:
  最高排名: 1
  最低排名: 56
  平均排名: 27.3
  中位排名: 27.0

排名分布:
  Top 10: 1 / 6 特征 (16.7%)
  Top 20: 2 / 6 特征 (33.3%)
  Top 30: 3 / 6 特征 (50.0%)

🔥 准备 Top 50 蜂群图 (Beeswarm Plot) 数据...